In [ ]:
!pip install -q wandb pytorch-msssim torchmetrics pytorch-lightning thop

import os
import random
from dataclasses import dataclass
from PIL import Image

import torch
import torch.fft
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader
from einops import rearrange
import pytorch_lightning as pl
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from pytorch_msssim import ms_ssim

import wandb

# Config

In [ ]:
@dataclass
class Config:
    data_dir: str = '/kaggle/input/datasets/tanhyml/lol-v2-dataset/LOL-v2' # Thay đổi nếu cần
    work_dir: str = '/kaggle/working'
    ckpt_dir: str = '/kaggle/working/checkpoints'
    # Cấu hình chuẩn của repo LYT-Net:
    patch_size: int = 256
    batch_size: int = 8
    learning_rate: float = 2e-4
    max_epochs: int = 500
    num_workers: int = 4
    seed: int = 1

    project_name: str = "LYT-Net_PyTorch"
    run_name: str = "new-1000e-LoLv2Real-OriLoss"

cfg = Config()

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("✅ Đã xác thực WandB thành công!")
except Exception as e:
    print("⚠️ Lỗi xác thực WandB. Chạy local mode.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: No netrc file found, creating one.


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


wandb: Currently logged in as: nghianguyen-21022006 (nghianguyen-21022006-i-h-c-qu-c-gia-tp-hcm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Đã xác thực WandB thành công!


# Dataset

In [ ]:
# class LOLDataset(Dataset):
#     def __init__(self, root_dir, mode='train', patch_size=256):
#         self.root_dir = root_dir
#         self.mode = mode
#         self.patch_size = patch_size

#         self.data_dir = os.path.join(root_dir, 'our485' if mode == 'train' else 'eval15')
#         self.low_dir = os.path.join(self.data_dir, 'low')
#         self.high_dir = os.path.join(self.data_dir, 'high')

#         self.image_names = sorted([f for f in os.listdir(self.low_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

#     def __len__(self):
#         return len(self.image_names)

#     def __getitem__(self, idx):
#         img_name = self.image_names[idx]
#         low_img = Image.open(os.path.join(self.low_dir, img_name)).convert('RGB')
#         high_img = Image.open(os.path.join(self.high_dir, img_name)).convert('RGB')

#         # Augmentation chuẩn theo bài báo (Crop 256, Flip, Rotate)
#         if self.mode == 'train':
#             w, h = low_img.size
#             if w >= self.patch_size and h >= self.patch_size:
#                 i, j, th, tw = T.RandomCrop.get_params(low_img, output_size=(self.patch_size, self.patch_size))
#                 low_img = TF.crop(low_img, i, j, th, tw)
#                 high_img = TF.crop(high_img, i, j, th, tw)

#             if random.random() > 0.5:
#                 low_img = TF.hflip(low_img)
#                 high_img = TF.hflip(high_img)

#             if random.random() > 0.5:
#                 low_img = TF.vflip(low_img)
#                 high_img = TF.vflip(high_img)

#             angle = random.choice([0, 90, 180, 270])
#             if angle != 0:
#                 low_img = TF.rotate(low_img, angle)
#                 high_img = TF.rotate(high_img, angle)

#         return TF.to_tensor(low_img), TF.to_tensor(high_img), img_name

In [ ]:
class LOLDataset(Dataset):
    def __init__(self, root_dir, mode='train', dataset_type='real', patch_size=256):
        """
        root_dir: Đường dẫn đến thư mục chứa 'Real_captured' và 'Synthetic' (thường là thư mục 'LOL-v2')
        mode: 'train' hoặc 'test'
        dataset_type: 'real' hoặc 'synthetic'
        """
        self.root_dir = root_dir
        self.mode = mode.lower()
        self.dataset_type = dataset_type.lower()
        self.patch_size = patch_size

        # Ánh xạ tên thư mục cha theo loại dataset
        type_folder = 'Real_captured' if self.dataset_type == 'real' else 'Synthetic'

        # Ánh xạ tên thư mục theo mode (chú ý viết hoa chữ cái đầu theo đúng ảnh)
        mode_folder = 'Train' if self.mode == 'train' else 'Test'

        # Xây dựng đường dẫn (VD: root_dir/Real_captured/Train)
        self.data_dir = os.path.join(self.root_dir, type_folder, mode_folder)

        # Tên thư mục chứa ảnh tối và sáng ở LOL-v2 là 'Low' và 'Normal'
        self.low_dir = os.path.join(self.data_dir, 'Low')
        self.high_dir = os.path.join(self.data_dir, 'Normal')

        self.image_names = sorted([f for f in os.listdir(self.low_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]

        # 1. Tạo tên file dự kiến cho ảnh High/Normal
        # (Ví dụ: 'low00690.png' -> 'normal00690.png')
        high_img_name = img_name.replace('low', 'normal')

        # 2. Xây dựng đường dẫn
        high_img_path = os.path.join(self.high_dir, high_img_name)

        # 3. Fallback: Đề phòng trường hợp dataset của bạn không đổi tên
        # (cả 2 bên đều tên giống nhau), ta check nếu file 'normal...' không tồn tại thì dùng tên gốc
        if not os.path.exists(high_img_path):
            high_img_path = os.path.join(self.high_dir, img_name)

        # 4. Load ảnh
        low_img = Image.open(os.path.join(self.low_dir, img_name)).convert('RGB')
        high_img = Image.open(high_img_path).convert('RGB')

        # Augmentation chuẩn theo bài báo (Crop 256, Flip, Rotate)
        if self.mode == 'train':
            w, h = low_img.size
            if w >= self.patch_size and h >= self.patch_size:
                i, j, th, tw = T.RandomCrop.get_params(low_img, output_size=(self.patch_size, self.patch_size))
                low_img = TF.crop(low_img, i, j, th, tw)
                high_img = TF.crop(high_img, i, j, th, tw)

            if random.random() > 0.5:
                low_img = TF.hflip(low_img)
                high_img = TF.hflip(high_img)

            if random.random() > 0.5:
                low_img = TF.vflip(low_img)
                high_img = TF.vflip(high_img)

            angle = random.choice([0, 90, 180, 270])
            if angle != 0:
                low_img = TF.rotate(low_img, angle)
                high_img = TF.rotate(high_img, angle)

        return TF.to_tensor(low_img), TF.to_tensor(high_img), img_name

In [ ]:
class LOLDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str, batch_size: int, num_workers: int, patch_size: int):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.patch_size = patch_size

    def setup(self, stage=None):
        self.train_dataset = LOLDataset(self.data_dir, mode='train', patch_size=self.patch_size)
        self.val_dataset = LOLDataset(self.data_dir, mode='eval', patch_size=self.patch_size)
        self.test_dataset = LOLDataset(self.data_dir, mode='eval', patch_size=self.patch_size)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=True,
            persistent_workers=True  # <--- THÊM VÀO ĐÂY
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=1,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True  # <--- THÊM VÀO ĐÂY
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=1,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True  # <--- THÊM VÀO ĐÂY
        )

# Loss

In [ ]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self, device):
        super(VGGPerceptualLoss, self).__init__()
        vgg = models.vgg19(weights='DEFAULT').features[:16]
        self.loss_model = vgg.to(device).eval()
        for param in self.loss_model.parameters():
            param.requires_grad = False

    def forward(self, y_true, y_pred):
        y_true, y_pred = y_true.to(next(self.loss_model.parameters()).device), y_pred.to(next(self.loss_model.parameters()).device)
        return F.mse_loss(self.loss_model(y_true), self.loss_model(y_pred))

def color_loss(y_true, y_pred):
    return torch.mean(torch.abs(torch.mean(y_true, dim=[1, 2, 3]) - torch.mean(y_pred, dim=[1, 2, 3])))


def smooth_l1_loss(y_true, y_pred):
    return F.smooth_l1_loss(y_true, y_pred)

def multiscale_ssim_loss(y_true, y_pred, max_val=1.0):
    return 1.0 - ms_ssim(y_true, y_pred, data_range=max_val, size_average=True)

def gaussian_kernel(x, mu, sigma):
    return torch.exp(-0.5 * ((x - mu) / sigma) ** 2)

def histogram_loss(y_true, y_pred, bins=256, sigma=0.01):
    bin_edges = torch.linspace(0.0, 1.0, bins, device=y_true.device)
    y_true_hist = torch.sum(gaussian_kernel(y_true.unsqueeze(-1), bin_edges, sigma), dim=0)
    y_pred_hist = torch.sum(gaussian_kernel(y_pred.unsqueeze(-1), bin_edges, sigma), dim=0)

    # SỬA Ở ĐÂY: Thêm 1e-8 vào mẫu số
    y_true_hist = y_true_hist / (y_true_hist.sum() + 1e-8)
    y_pred_hist = y_pred_hist / (y_pred_hist.sum() + 1e-8)

    hist_distance = torch.mean(torch.abs(y_true_hist - y_pred_hist))
    return hist_distance

def psnr_loss(y_true, y_pred):
    mse = F.mse_loss(y_true, y_pred)
    # SỬA Ở ĐÂY: Thêm 1e-8 vào mẫu số và bên trong log10
    psnr = 20 * torch.log10(1.0 / (torch.sqrt(mse) + 1e-8))
    return 40.0 - torch.mean(psnr)

class CombinedLoss(nn.Module):
    def __init__(self, device):
        super(CombinedLoss, self).__init__()
        self.perceptual_loss_model = VGGPerceptualLoss(device)
        self.alpha1 = 1.00
        self.alpha2 = 0.06
        self.alpha3 = 0.05
        self.alpha4 = 0.5
        self.alpha5 = 0.0083
        self.alpha6 = 0.25

    def forward(self, y_true, y_pred):
        smooth_l1_l = smooth_l1_loss(y_true, y_pred)
        ms_ssim_l = multiscale_ssim_loss(y_true, y_pred)
        perc_l = self.perceptual_loss_model(y_true, y_pred)
        hist_l = histogram_loss(y_true, y_pred)
        psnr_l = psnr_loss(y_true, y_pred)
        color_l = color_loss(y_true, y_pred)

        total_loss = (self.alpha1 * smooth_l1_l + self.alpha2 * perc_l +
                      self.alpha3 * hist_l + self.alpha5 * psnr_l +
                      self.alpha6 * color_l + self.alpha4 * ms_ssim_l)
        return torch.mean(total_loss)

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# from pytorch_msssim import ms_ssim

# # 1. Charbonnier Loss (Phục hồi cấu trúc pixel sắc nét)
# class CharbonnierLoss(nn.Module):
#     def __init__(self, eps=1e-3):
#         super(CharbonnierLoss, self).__init__()
#         self.eps = eps

#     def forward(self, x, y):
#         diff = x - y
#         return torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))

# # 2. VGG Perceptual Loss (Đảm bảo độ tự nhiên cho mắt người)
# class VGGPerceptualLoss(nn.Module):
#     def __init__(self, device):
#         super(VGGPerceptualLoss, self).__init__()
#         vgg = models.vgg19(weights='DEFAULT').features[:36]
#         self.loss_model = vgg.to(device).eval()
#         for param in self.loss_model.parameters():
#             param.requires_grad = False

#     def forward(self, y_true, y_pred):
#         return F.l1_loss(self.loss_model(y_true), self.loss_model(y_pred))

# # 3. MỚI: Cosine Color Loss (Ép màu sắc rực rỡ và đúng tông)
# class CosineColorLoss(nn.Module):
#     def __init__(self):
#         super(CosineColorLoss, self).__init__()

#     def forward(self, y_pred, y_true):
#         # Tính tích vô hướng (Dot product) giữa vector RGB của Pred và True dọc theo trục Channel (dim=1)
#         dot_product = (y_pred * y_true).sum(dim=1)

#         # Tính độ lớn (Norm) của vector RGB
#         norm_pred = torch.sqrt((y_pred * y_pred).sum(dim=1) + 1e-8)
#         norm_true = torch.sqrt((y_true * y_true).sum(dim=1) + 1e-8)

#         # Tính Cosine Similarity: cos = (A.B) / (|A|*|B|)
#         cos_sim = dot_product / (norm_pred * norm_true)

#         # Vì ta muốn tối thiểu hóa Loss, nên lấy 1 trừ đi trung bình Cosine Similarity
#         # cos_sim tiến tới 1 -> loss tiến tới 0 (Hai màu cùng hướng)
#         return 1.0 - cos_sim.mean()

# # ==========================================
# # KHỐI LOSS TỔNG HỢP (FINAL COMBO)
# # ==========================================
# class CombinedLoss(nn.Module):
#     def __init__(self, device):
#         super(CombinedLoss, self).__init__()
#         self.charbonnier = CharbonnierLoss()
#         self.perceptual = VGGPerceptualLoss(device)
#         self.color_loss = CosineColorLoss()

#         # BỘ TRỌNG SỐ ĐỀ XUẤT (Đã được chứng minh hiệu quả trên LOL dataset)
#         self.w_char = 1.0       # Kéo PSNR
#         self.w_ssim = 0.2       # Kéo SSIM, giữ chi tiết
#         self.w_perceptual = 0.1 # Giữ texture tự nhiên
#         self.w_color = 0.5      # Bơm màu sắc rực rỡ và bù sáng

#     def forward(self, y_pred, y_true):
#         # Pixel Loss
#         l_char = self.charbonnier(y_pred, y_true)

#         # SSIM Loss
#         l_ssim = 1.0 - ms_ssim(y_pred, y_true, data_range=1.0, size_average=True)

#         # Perceptual Loss
#         l_perc = self.perceptual(y_true, y_pred)

#         # Color Loss
#         l_color = self.color_loss(y_pred, y_true)

#         total_loss = (self.w_char * l_char) + \
#                      (self.w_ssim * l_ssim) + \
#                      (self.w_perceptual * l_perc) + \
#                      (self.w_color * l_color)

#         return total_loss

# Arch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

# ==============================================================================
# HÀM HỖ TRỢ: BIẾN ĐỔI HAAR WAVELET SIÊU TỐC [THÊM MỚI]
# ==============================================================================
# def get_haar_dwt(x):
#     """Biến đổi Wavelet (Băm thành 4 dải tần) bằng Slicing Ma trận"""
#     x01 = x[:, :, 0::2, 1::2]
#     x10 = x[:, :, 1::2, 0::2]
#     x11 = x[:, :, 1::2, 1::2]
#     x00 = x[:, :, 0::2, 0::2]

#     ll = (x00 + x01 + x10 + x11) * 0.5
#     hl = (x00 - x01 + x10 - x11) * 0.5
#     lh = (x00 + x01 - x10 - x11) * 0.5
#     hh = (x00 - x01 - x10 + x11) * 0.5

#     return torch.cat([ll, hl, lh, hh], dim=1)

# def get_haar_iwt(x):
#     """Biến đổi Wavelet Ngược (Ráp 4 dải tần về Không gian)"""
#     B, C, H, W = x.shape
#     out_C = C // 4

#     ll, hl, lh, hh = torch.chunk(x, 4, dim=1)

#     x00 = (ll + hl + lh + hh) * 0.5
#     x01 = (ll - hl + lh - hh) * 0.5
#     x10 = (ll + hl - lh - hh) * 0.5
#     x11 = (ll - hl - lh + hh) * 0.5

#     out = torch.empty((B, out_C, H * 2, W * 2), dtype=x.dtype, device=x.device)
#     out[:, :, 0::2, 0::2] = x00
#     out[:, :, 0::2, 1::2] = x01
#     out[:, :, 1::2, 0::2] = x10
#     out[:, :, 1::2, 1::2] = x11

#     return out

# ==============================================================================
# KHỐI GMWTConvs TỐI ƯU SIÊU NHẸ [THÊM MỚI]
# ==============================================================================
# class LightweightGMWTConv(nn.Module):
#     """Khối trích xuất đặc trưng Wavelet từ ảnh đầu vào để làm Prior"""
#     def __init__(self, in_channels, out_channels):
#         super().__init__()

#         self.shallow = nn.Sequential(
#             nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels),
#             nn.GELU()
#         )

#         wt_channels = in_channels * 4
#         self.freq_conv = nn.Sequential(
#             nn.Conv2d(wt_channels, wt_channels, kernel_size=3, padding=1, groups=wt_channels),
#             nn.GELU(),
#             nn.Conv2d(wt_channels, wt_channels, kernel_size=1)
#         )

#         # self.out_proj = nn.Conv2d(in_channels, out_channels, kernel_size=1)
#         self.out_proj = nn.Sequential(
#             # 1. Standard Conv: Trộn cả không gian lẫn kênh (Đóng vai trò chính để XÓA LỖI CHECKERBOARD)
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.GELU(),

#             # 2. Depthwise Conv: Phân tích sâu hơn từng kênh độc lập (Tăng độ chính xác, tốn rất ít Params/FLOPs)
#             nn.Conv2d(out_channels, 2 * out_channels, kernel_size=3, padding=1, groups=out_channels),
#             nn.GELU(),

#             # 3. Pointwise Conv: Nén số kênh từ 2*out_channels về lại out_channels (KHÔNG dùng groups ở đây)
#             nn.Conv2d(2 * out_channels, out_channels, kernel_size=1)
#         )

#     def forward(self, x):
#         x_shallow = self.shallow(x)
#         x_wt = get_haar_dwt(x_shallow)
#         x_wt_feat = self.freq_conv(x_wt)
#         x_iwt = get_haar_iwt(x_wt_feat)
#         return self.out_proj(x_iwt)

class LightweightPriorExtractor(nn.Module):
    """
    Khối trích xuất đặc trưng cục bộ (Cạnh, cấu trúc) siêu nhẹ
    THAY THẾ HOÀN TOÀN CHO GMWT VÀ WAVELET PRIOR.
    """
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()

        # Bước 1: Trích xuất đặc trưng không gian (như cạnh, viền)
        # Sử dụng kernel 3x3 để quét qua ảnh đầu vào.
        self.spatial_extract = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * 4, kernel_size=3, padding=1),
            LayerNorm2d(in_channels * 4),
            nn.GELU()
        )

        # Bước 2: Xử lý sâu hơn với chi phí cực thấp bằng Depthwise Convolution
        hidden_dim = in_channels * 4
        self.depthwise_process = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim),
            LayerNorm2d(hidden_dim),
            nn.GELU()
        )

        # Bước 3: Ép số kênh về đúng out_channels để chuẩn bị Concat với nhánh chính
        self.channel_proj = nn.Conv2d(hidden_dim, out_channels, kernel_size=1)

    def forward(self, x):
        feat = self.spatial_extract(x)
        feat = self.depthwise_process(feat)
        out = self.channel_proj(feat)
        return out


# ==============================================================================
# 1. CÁC MODULE CƠ BẢN (TIỆN ÍCH)
# ==============================================================================
class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class LayerNorm2d(nn.Module):
    """Sử dụng LayerNorm thay cho BatchNorm2d để ổn định với Batch Size nhỏ"""
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        return self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


# ==============================================================================
# 2. NHÁNH GLOBAL (ATTENTION DỰA TRÊN CỬA SỔ VÀ KÊNH)
# ==============================================================================
class GlobalAttentionBlock(nn.Module):
    """Trích xuất ngữ cảnh toàn cục qua Spatial & Channel Attention, có DWConv tinh chỉnh Artifact"""
    def __init__(self, dim, window_size=8):
        super().__init__()
        self.dim = dim
        self.window_size = window_size

        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=False)
        self.temperature = nn.Parameter(torch.ones(1, 1, 1))

        self.smooth_conv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)

        self.proj = nn.Conv2d(dim, dim, kernel_size=1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape

        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x = F.pad(x, (0, pad_r, 0, pad_b))
            _, _, H_pad, W_pad = x.shape
        else:
            H_pad, W_pad = H, W

        q, k, v = self.qkv(x).chunk(3, dim=1)

        # 1. Channel-Wise Self-Attention
        q_c, k_c, v_c = q.view(B, C, -1), k.view(B, C, -1), v.view(B, C, -1)
        scale_c = (H_pad * W_pad) ** -0.5
        attn_c = (q_c @ k_c.transpose(-2, -1)) * self.temperature * scale_c
        attn_c = attn_c.softmax(dim=-1)
        y_c = (attn_c @ v_c).view(B, C, H_pad, W_pad)

        # 2. Spatial Window Self-Attention
        q_s = q.view(B, C, H_pad // self.window_size, self.window_size, W_pad // self.window_size, self.window_size)
        q_s = q_s.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, self.window_size**2, C)

        k_s = k.view(B, C, H_pad // self.window_size, self.window_size, W_pad // self.window_size, self.window_size)
        k_s = k_s.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, self.window_size**2, C)

        v_s = v.view(B, C, H_pad // self.window_size, self.window_size, W_pad // self.window_size, self.window_size)
        v_s = v_s.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, self.window_size**2, C)

        attn_s = (q_s @ k_s.transpose(-2, -1)) * (C ** -0.5)
        y_s = (attn_s.softmax(dim=-1) @ v_s)

        y_s = y_s.view(B, H_pad // self.window_size, W_pad // self.window_size, self.window_size, self.window_size, C)
        y_s = y_s.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, C, H_pad, W_pad)

        y_s = self.smooth_conv(y_s)

        # 3. Adaptive Interaction
        y_s_hat = y_s * torch.sigmoid(y_c)
        y_c_hat = y_c * torch.sigmoid(y_s)

        out = self.proj(y_s_hat + y_c_hat)

        if pad_r > 0 or pad_b > 0:
            out = out[:, :, :H, :W]

        return out


class GlobalEncoderStage(nn.Module):
    def __init__(self, in_dim, out_dim, window_size=8, apply_downsample=True):
        super().__init__()
        if apply_downsample:
            self.downsample = nn.Conv2d(in_dim, out_dim, kernel_size=4, stride=2, padding=1)
        else:
            self.downsample = nn.Identity()
            out_dim = in_dim

        self.norm1 = LayerNorm2d(out_dim)
        self.attention = GlobalAttentionBlock(out_dim, window_size)

        self.norm2 = LayerNorm2d(out_dim)
        self.ffn = nn.Sequential(
            nn.Conv2d(out_dim, out_dim * 2, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(out_dim * 2, out_dim, kernel_size=1)
        )

    def forward(self, x):
        x = self.downsample(x)
        x = self.attention(self.norm1(x)) + x
        x = self.ffn(self.norm2(x)) + x
        return x


# ==============================================================================
# 3. NHÁNH LOCAL (TRÍCH XUẤT ĐẶC TRƯNG ĐỊA PHƯƠNG BẰNG TÍCH CHẬP)
# ==============================================================================
class LocalInceptionBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.branch1 = nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1, groups=dim, bias=False), LayerNorm2d(dim))
        self.branch2 = nn.Sequential(nn.Conv2d(dim, dim, (3, 1), padding=(1, 0), groups=dim, bias=False), LayerNorm2d(dim))
        self.branch3 = nn.Sequential(nn.Conv2d(dim, dim, (1, 3), padding=(0, 1), groups=dim, bias=False), LayerNorm2d(dim))
        self.branch4 = nn.Sequential(nn.Conv2d(dim, dim, 1, bias=False), LayerNorm2d(dim))
        self.fusion = nn.Conv2d(dim * 4, dim, kernel_size=1, bias=False)

    def forward(self, x):
        out = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)
        return self.fusion(out)

class LocalFeatureStage(nn.Module):
    def __init__(self, in_dim, out_dim, apply_downsample=True):
        super().__init__()
        if apply_downsample:
            self.downsample = nn.Conv2d(in_dim, out_dim, kernel_size=4, stride=2, padding=1)
        else:
            self.downsample = nn.Identity()
            out_dim = in_dim

        self.norm1 = LayerNorm2d(out_dim)
        self.local_extract = LocalInceptionBlock(out_dim)

        self.norm2 = LayerNorm2d(out_dim)
        self.ffn = nn.Sequential(
            nn.Conv2d(out_dim, out_dim * 2, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(out_dim * 2, out_dim, kernel_size=1)
        )

    def forward(self, x):
        x = self.downsample(x)
        x = self.local_extract(self.norm1(x)) + x
        x = self.ffn(self.norm2(x)) + x
        return x


# ==============================================================================
# 4. BOTTLENECK FUSION (DUNG HỢP THÔNG TIN TOÀN CỤC VÀ ĐỊA PHƯƠNG)
# ==============================================================================
class CrossAttentionModule(nn.Module):
    def __init__(self, embed_size, num_heads):
        super().__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads

        self.query_dense = nn.Linear(embed_size, embed_size)
        self.key_dense = nn.Linear(embed_size, embed_size)
        self.value_dense = nn.Linear(embed_size, embed_size)
        self.combine_heads = nn.Linear(embed_size, embed_size)

        init.xavier_uniform_(self.query_dense.weight); init.constant_(self.query_dense.bias, 0)
        init.xavier_uniform_(self.key_dense.weight); init.constant_(self.key_dense.bias, 0)
        init.xavier_uniform_(self.value_dense.weight); init.constant_(self.value_dense.bias, 0)
        init.xavier_uniform_(self.combine_heads.weight); init.constant_(self.combine_heads.bias, 0)

    def split_heads(self, x, batch_size):
        x = x.reshape(batch_size, -1, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def forward(self, x_query, x_kv):
        B, C, H, W = x_query.size()
        x_q = x_query.flatten(2).transpose(1, 2)
        x_k_v = x_kv.flatten(2).transpose(1, 2)

        query = self.split_heads(self.query_dense(x_q), B)
        key = self.split_heads(self.key_dense(x_k_v), B)
        value = self.split_heads(self.value_dense(x_k_v), B)

        attn_weights = F.softmax(torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5), dim=-1)
        attention = torch.matmul(attn_weights, value).permute(0, 2, 1, 3).contiguous().reshape(B, -1, self.embed_size)

        output = self.combine_heads(attention)
        return output.transpose(1, 2).reshape(B, C, H, W)

class CrossGatingFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj1 = nn.Conv2d(dim, dim, 1)
        self.dwconv1 = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm1 = LayerNorm2d(dim)

        self.proj2 = nn.Conv2d(dim, dim, 1)
        self.dwconv2 = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm2 = LayerNorm2d(dim)

        self.out_proj = nn.Conv2d(dim, dim, 1)

    def forward(self, x1, x2):
        z1 = self.norm1(self.dwconv1(self.proj1(x1)))
        z2 = self.norm2(self.dwconv2(self.proj2(x2)))
        return self.out_proj((z1 * F.gelu(z2)) + (z2 * torch.sigmoid(z1)))

class GlobalLocalFusionModule(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm_local = LayerNorm2d(dim)
        self.norm_global = LayerNorm2d(dim)

        self.cross_attention = CrossAttentionModule(dim, num_heads)
        self.cross_gating = CrossGatingFusion(dim)

    def forward(self, x_local, x_global):
        x_cross = self.cross_attention(x_query=self.norm_local(x_local),
                                       x_kv=self.norm_global(x_global))

        out = self.cross_gating(x_local, x_cross)
        return out + x_local


# ==============================================================================
# 5. REFINEMENT CUỐI (TINH CHỈNH ẢNH TRƯỚC KHI XUẤT RA)
# ==============================================================================
class NAFBlock(nn.Module):
    def __init__(self, c, DW_Expand=2, FFN_Expand=2):
        super().__init__()
        dw_channel = c * DW_Expand

        self.norm1 = LayerNorm2d(c)
        self.conv1 = nn.Conv2d(c, dw_channel, 1)
        self.conv2 = nn.Conv2d(dw_channel, dw_channel, 3, 1, 1, groups=dw_channel)
        self.sg = SimpleGate()
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw_channel // 2, dw_channel // 2, 1))
        self.conv3 = nn.Conv2d(dw_channel // 2, c, 1)

        self.norm2 = LayerNorm2d(c)
        ffn_channel = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn_channel, 1)
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, 1)

        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv3(self.sg(self.conv2(self.conv1(x))) * self.sca(self.sg(self.conv2(self.conv1(x)))))
        y = inp + x * self.beta

        x = self.conv5(self.sg(self.conv4(self.norm2(y))))
        return y + x * self.gamma

class ImageRefinementModule(nn.Module):
    def __init__(self, in_size=32, out_size=3, dim=32, num_blocks=2):
        super().__init__()
        self.intro = nn.Conv2d(in_size, dim, kernel_size=3, padding=1)
        self.body = nn.Sequential(*[NAFBlock(c=dim) for _ in range(num_blocks)])
        self.outro = nn.Conv2d(dim, out_size, kernel_size=3, padding=1)

    def forward(self, x):
        feat = self.intro(x)
        feat_refined = self.body(feat) + feat
        return torch.sigmoid(self.outro(feat_refined))


# ==============================================================================
# 6. MÔ HÌNH CHÍNH (DUAL-BRANCH U-NET)
# ==============================================================================``
class LightweightDualBranchUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_dim=16, num_heads=2, window_size=8):
        super().__init__()

        self.stem = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)

        # --- ENCODER ---
        self.enc_g1 = GlobalEncoderStage(base_dim, base_dim * 2, window_size, apply_downsample=True)
        self.enc_l1 = LocalFeatureStage(base_dim, base_dim * 2, apply_downsample=True)

        self.enc_g2 = GlobalEncoderStage(base_dim * 2, base_dim * 4, window_size, apply_downsample=True)
        self.enc_l2 = LocalFeatureStage(base_dim * 2, base_dim * 4, apply_downsample=True)

        # ----------------------------------------------------------------------
        # [THÊM MỚI] WAVELET PRIOR & FUSION LAYERS
        # ----------------------------------------------------------------------
        # GMWT cho Stage 1: Nhận ảnh RGB (3 kênh), xuất ra số kênh = đầu ra của enc_1 (base_dim * 2)
        # self.gmwt_s1 = LightweightGMWTConv(in_channels=3, out_channels=base_dim * 2)
        self.prior_s1 = LightweightPriorExtractor(in_channels=3, out_channels=base_dim * 2)
        # Các lớp Conv để hòa trộn và ép kênh sau khi Concat
        self.reduce_s1_g = nn.Conv2d(base_dim * 2 * 2, base_dim * 2, kernel_size=1)
        self.reduce_s1_l = nn.Conv2d(base_dim * 2 * 2, base_dim * 2, kernel_size=1)

        # GMWT cho Stage 2: Nhận ảnh RGB (3 kênh), xuất ra số kênh = đầu ra của enc_2 (base_dim * 4)
        # self.gmwt_s2 = LightweightGMWTConv(in_channels=3, out_channels=base_dim * 4)
        self.prior_s2 = LightweightPriorExtractor(in_channels=3, out_channels=base_dim * 4)
        self.reduce_s2_g = nn.Conv2d(base_dim * 4 * 2, base_dim * 4, kernel_size=1)
        self.reduce_s2_l = nn.Conv2d(base_dim * 4 * 2, base_dim * 4, kernel_size=1)
        # ----------------------------------------------------------------------

        # --- BOTTLENECK FUSION ---
        self.fusion = GlobalLocalFusionModule(base_dim * 4, num_heads)

        # --- DECODER ---

        # Chúng ta sẽ concat: fused_s4 (base_dim*4), g2 (base_dim*4), l2 (base_dim*4)
        # Tổng cộng đầu vào sẽ là base_dim * 12
        self.reduce_s3 = nn.Conv2d(base_dim * 12, base_dim * 4, kernel_size=1)

        # 2. Khối xử lý đặc trưng tại scale s/4 (Decoder Stage 3)
        self.dec_s3 = LocalFeatureStage(base_dim * 4, base_dim * 4, apply_downsample=False)

        self.up_s2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=3, padding=1)
        )
        self.reduce_s2 = nn.Conv2d(base_dim * 6, base_dim * 2, kernel_size=1)
        self.dec_s2 = LocalFeatureStage(base_dim * 2, base_dim * 2, apply_downsample=False)

        self.up_s = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(base_dim * 2, base_dim, kernel_size=3, padding=1)
        )
        self.reduce_s = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec_s = LocalFeatureStage(base_dim, base_dim, apply_downsample=False)

        # --- REFINEMENT ---
        self.refinement = ImageRefinementModule(in_size=base_dim, out_size=out_channels, dim=base_dim, num_blocks=4)

    def forward(self, x):
        feat_s = self.stem(x)

        # ==========================================
        # STAGE 1: ENCODER & WAVELET INJECTION
        # ==========================================
        g1, l1 = self.enc_g1(feat_s), self.enc_l1(feat_s)

        # [THÊM MỚI] Tạo ảnh downsample để khớp kích thước H/2, W/2 với đầu ra của Stage 1
        x_down1 = F.interpolate(x, size=g1.shape[2:], mode='bilinear', align_corners=False)
        prior_1 = self.prior_s1(x_down1)

        # Nối (Concat) Prior vào Nhánh Global & Local rồi Ép kênh
        g1 = self.reduce_s1_g(torch.cat([g1, prior_1], dim=1))
        l1 = self.reduce_s1_l(torch.cat([l1, prior_1], dim=1))

        # ==========================================
        # STAGE 2: ENCODER & WAVELET INJECTION
        # ==========================================
        g2, l2 = self.enc_g2(g1), self.enc_l2(l1)

        # [THÊM MỚI] Tạo ảnh downsample để khớp kích thước H/4, W/4 với đầu ra của Stage 2
        x_down2 = F.interpolate(x, size=g2.shape[2:], mode='bilinear', align_corners=False)
        prior_2 = self.prior_s2(x_down2)

        # Nối (Concat) Prior vào Nhánh Global & Local rồi Ép kênh
        g2 = self.reduce_s2_g(torch.cat([g2, prior_2], dim=1))
        l2 = self.reduce_s2_l(torch.cat([l2, prior_2], dim=1))

        # --- BOTTLENECK FUSION ---
        fused_s4 = self.fusion(x_local=l2, x_global=g2)

        # --- DECODER ---
        d3 = self.dec_s3(self.reduce_s3(torch.cat([fused_s4, g2, l2], dim=1)))
        d2 = self.up_s2(d3)
        if d2.shape[2:] != g1.shape[2:]: d2 = F.interpolate(d2, size=g1.shape[2:], mode='bilinear', align_corners=False)

        d2 = self.dec_s2(self.reduce_s2(torch.cat([d2, g1, l1], dim=1)))
        d_s = self.up_s(d2)
        if d_s.shape[2:] != feat_s.shape[2:]: d_s = F.interpolate(d_s, size=feat_s.shape[2:], mode='bilinear', align_corners=False)

        d_s = self.dec_s(self.reduce_s(torch.cat([d_s, feat_s], dim=1)))

        return self.refinement(d_s)

# ==============================================================================
# TEST & PROFILING MÔ HÌNH
# ==============================================================================
if __name__ == '__main__':
    import time
    from thop import profile, clever_format # [THÊM MỚI] Import thư viện thop

    indim = 256
    print(f"--- ĐANG TEST MÔ HÌNH VỚI KÍCH THƯỚC ẢNH: {indim}x{indim} ---")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LightweightDualBranchUNet(base_dim=18, num_heads=2).to(device)
    model.eval()
    dummy_input = torch.randn(1, 3, indim, indim).to(device)

    # 1. Đếm Parameters và FLOPS bằng THOP [THÊM MỚI]
    print("\nĐang tính toán Độ phức tạp (FLOPS/Params)...")
    with torch.no_grad():
        # Tham số verbose=False để ẩn các log chi tiết của từng layer
        macs, params = profile(model, inputs=(dummy_input, ), verbose=False)

    # Tính Giga (tỷ) để hiển thị
    G_MACs = macs / 1e9
    M_Params = params / 1e6
    G_FLOPs = (macs * 2) / 1e9 # 1 MAC = 2 FLOPs (1 nhân + 1 cộng)

    print(f"Tổng số tham số (Params): {M_Params:.3f} M")
    print(f"Tổng số MACs (Multiply-Accumulates): {G_MACs:.3f} G")
    print(f"Tổng số FLOPs thực tế: {G_FLOPs:.3f} G")

    # Đo Inference Time cơ bản
    print("\nĐang chạy test suy luận...")
    start_time = time.time()
    with torch.no_grad():
        out = model(dummy_input)
    end_time = time.time()

    print(f"✅ Hoàn tất! Kích thước đầu ra: {out.shape}")
    print(f"⏱ Thời gian Inference: {(end_time - start_time)*1000:.2f} ms")

--- ĐANG TEST MÔ HÌNH VỚI KÍCH THƯỚC ẢNH: 256x256 ---



Đang tính toán Độ phức tạp (FLOPS/Params)...


Tổng số tham số (Params): 0.416 M
Tổng số MACs (Multiply-Accumulates): 4.575 G
Tổng số FLOPs thực tế: 9.149 G

Đang chạy test suy luận...
✅ Hoàn tất! Kích thước đầu ra: torch.Size([1, 3, 256, 256])
⏱ Thời gian Inference: 12.36 ms


In [ ]:
# import torch
# import torch.nn.functional as F
# import matplotlib.pyplot as plt
# import numpy as np
# from PIL import Image
# import torchvision.transforms as T
# import os

# # ==============================================================================
# # 1. CÔNG CỤ TRÍCH XUẤT ĐẶC TRƯNG (FEATURE EXTRACTOR)
# # ==============================================================================
# class FeatureExtractor:
#     def __init__(self, model, layers_to_monitor):
#         self.features = {}
#         self.hooks = []
#         for name, layer in layers_to_monitor.items():
#             hook = layer.register_forward_hook(self._save_activation(name))
#             self.hooks.append(hook)

#     def _save_activation(self, name):
#         def hook(module, input, output):
#             if isinstance(output, tuple):
#                 output = output[0]
#             self.features[name] = output.detach().cpu()
#         return hook

#     def remove_hooks(self):
#         for hook in self.hooks:
#             hook.remove()

# # ==============================================================================
# # 2. HÀM TRỰC QUAN HÓA CHỒNG ẢNH (OVERLAY HEATMAP)
# # ==============================================================================
# def analyze_and_plot_overlay(feature_tensor, layer_name, orig_image_tensor):
#     # Lấy tensor đặc trưng của ảnh đầu tiên trong batch
#     feat = feature_tensor[0:1]

#     # [SỬA QUAN TRỌNG] Lấy giá trị tuyệt đối trước khi tính trung bình
#     # để tránh các giá trị âm/dương tự triệt tiêu nhau làm sai lệch Heatmap
#     mean_act = torch.mean(torch.abs(feat), dim=1, keepdim=True)

#     # Resize Heatmap về đúng bằng kích thước ảnh gốc (256x256)
#     _, _, H_orig, W_orig = orig_image_tensor.shape
#     mean_act_resized = F.interpolate(mean_act, size=(H_orig, W_orig), mode='bilinear', align_corners=False)

#     # Chuẩn hóa Heatmap về [0, 1]
#     heatmap = mean_act_resized[0, 0].numpy()
#     if heatmap.max() - heatmap.min() > 0:
#         heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())

#     # Xử lý ảnh gốc để vẽ bằng Matplotlib
#     orig_img_np = orig_image_tensor[0].cpu().permute(1, 2, 0).numpy()
#     orig_img_np = np.clip((orig_img_np - orig_img_np.min()) / (orig_img_np.max() - orig_img_np.min() + 1e-8), 0, 1)

#     # Thiết lập khung vẽ 3 cột
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#     fig.suptitle(f'Phân tích Layer: {layer_name}', fontsize=16, fontweight='bold', y=1.05)

#     # Cột 1: Ảnh gốc
#     axes[0].imshow(orig_img_np)
#     axes[0].set_title("1. Ảnh đầu vào (Low-light)", fontsize=12)
#     axes[0].axis('off')

#     # Cột 2: Heatmap
#     im2 = axes[1].imshow(heatmap, cmap='jet')
#     axes[1].set_title("2. Mức độ kích hoạt (Heatmap)", fontsize=12)
#     axes[1].axis('off')

#     # Cột 3: Overlay (Chồng ảnh gốc và Heatmap)
#     axes[2].imshow(orig_img_np)
#     axes[2].imshow(heatmap, cmap='jet', alpha=0.55) # alpha=0.55 tạo độ trong suốt
#     axes[2].set_title("3. Chồng lên nhau (Overlay)", fontsize=12)
#     axes[2].axis('off')

#     plt.tight_layout()
#     plt.show()

# # ==============================================================================
# # 3. KHỞI TẠO, LOAD CHECKPOINT VÀ CHẠY MÔ HÌNH
# # ==============================================================================
# # Đảm bảo class LightweightDualBranchUNet đã được chạy ở cell trên

# checkpoint_path = '/kaggle/input/datasets/vgbao312/finalmodel/lytnet-epoch398-val_psnr23.25.ckpt'
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = LightweightDualBranchUNet(base_dim=18, num_heads=2).to(device)

# print(f"Đang load checkpoint từ: {checkpoint_path}...")
# checkpoint = torch.load(checkpoint_path, map_location=device)

# # --- [SỬA QUAN TRỌNG] Tái sử dụng logic làm sạch State Dict mạnh mẽ nhất ---
# if 'state_dict' in checkpoint:
#     state_dict = checkpoint['state_dict']
# else:
#     state_dict = checkpoint

# clean_state_dict = {}
# for k, v in state_dict.items():
#     # Bỏ qua các trọng số của hàm loss
#     if k.startswith('lpips') or k.startswith('criterion'):
#         continue

#     clean_key = k
#     if clean_key.startswith('model.'): clean_key = clean_key.replace('model.', '', 1)
#     elif clean_key.startswith('net.'): clean_key = clean_key.replace('net.', '', 1)
#     if clean_key.startswith('_orig_mod.'): clean_key = clean_key.replace('_orig_mod.', '', 1)

#     clean_state_dict[clean_key] = v

# try:
#     # strict=False giúp bỏ qua các key thừa một cách êm ái
#     model.load_state_dict(clean_state_dict, strict=False)
#     print("✅ Load checkpoint thành công!")
# except RuntimeError as e:
#     print("❌ Lỗi khi map state_dict. Chi tiết:", e)
# # -------------------------------------------------------------------------

# model.eval()

# # Định nghĩa các layer cần theo dõi (Bổ sung 2 lớp GMWT để phân tích)
# layers_to_monitor = {
#     '01_Stem': model.stem,
#     '02_Global_Encoder_1': model.enc_g1,
#     '03_Local_Encoder_1': model.enc_l1,
#     '03b_Wavelet_Prior_1': model.prior_s1,  # Đã thêm
#     '04_Global_Encoder_2': model.enc_g2,
#     '05_Local_Encoder_2': model.enc_l2,
#     '05b_Wavelet_Prior_2': model.prior_s2,  # Đã thêm
#     '06_Bottleneck_Fusion': model.fusion,
#     '07_Decoder_Stage_2': model.dec_s2,
#     '08_Decoder_Stage_1': model.dec_s,
#     '09_Output_Refinement': model.refinement
# }

# extractor = FeatureExtractor(model, layers_to_monitor)

# # ==============================================================================
# # 4. ĐƯA ẢNH THẬT VÀO VÀ TRỰC QUAN HÓA
# # ==============================================================================
# img_path = "/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15/low/748.png"
# print(f"\nĐang load ảnh từ: {img_path}...")

# if not os.path.exists(img_path):
#     print(f"❌ KHÔNG TÌM THẤY ẢNH TẠI ĐƯỜNG DẪN: {img_path}")
#     print("Hãy đảm bảo bạn đã 'Add Data' thư mục lol-dataset vào notebook của mình.")
# else:
#     original_pil = Image.open(img_path).convert('RGB')

#     # Tiền xử lý ảnh: Đưa về 256x256 và chuyển thành Tensor
#     transform = T.Compose([
#         T.Resize((256, 256)),
#         T.ToTensor()
#     ])
#     real_image = transform(original_pil).unsqueeze(0).to(device)
#     print("✅ Đã load ảnh và sẵn sàng suy luận!")

#     # Chạy ảnh qua mô hình
#     with torch.no_grad():
#         _ = model(real_image)

#     # In ra biểu đồ Overlay
#     print("\n--- BẮT ĐẦU VẼ FEATURE MAPS OVERLAY ---")
#     for layer_name, feature_tensor in sorted(extractor.features.items()):
#         analyze_and_plot_overlay(feature_tensor, layer_name, real_image)

# # Dọn dẹp RAM/VRAM
# extractor.remove_hooks()
# print("✅ Quá trình trực quan hóa hoàn tất!")

# Lightning module

In [ ]:
# import os
# import torch
# import torch.optim as optim
# import pytorch_lightning as pl
# import torchvision.utils as vutils
# # Nếu dùng wandb
# try:
#     import wandb
# except ImportError:
#     pass

# from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
# # THÊM IMPORT CHO LPIPS
# from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

# class LYTLightning(pl.LightningModule):
#     def __init__(self, learning_rate=2e-4, max_epochs=1000):
#         super().__init__()
#         self.save_hyperparameters()

#         self.model = LightweightDualBranchUNet(base_dim=18)
#         # Loss Model cần device, nên ta khởi tạo lazy hoặc để hook vào Lightning
#         self.criterion = None

#         self.psnr = PeakSignalNoiseRatio(data_range=1.0)
#         self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0)

#         # THÊM LPIPS
#         # Thông thường net_type = 'vgg' hoặc 'alex' được dùng phổ biến
#         # normalize=True giả định input vào khoảng [0, 1] (giống psnr/ssim data_range)
#         self.lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True)

#     def on_fit_start(self):
#         # Khởi tạo loss sau khi module đã được map lên đúng thiết bị GPU
#         self.criterion = CombinedLoss(self.device)

#     def forward(self, x):
#         return self.model(x)

#     def configure_optimizers(self):
#         optimizer = optim.Adam(self.model.parameters(), lr=self.hparams.learning_rate)
#         scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs)
#         return {
#             "optimizer": optimizer,
#             "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}
#         }

#     def training_step(self, batch, batch_idx):
#         low_imgs, high_imgs, img_names = batch
#         enhanced_imgs = self.forward(low_imgs)

#         # Loss có giám sát: Giữa ảnh được dự đoán và ảnh GroundTruth (high_imgs)
#         loss = self.criterion(high_imgs, enhanced_imgs)

#         self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#         return loss

#     # === Logic chuẩn GT-MEAN của LOL Dataset (Theo test.py repo gốc) ===
#     def apply_gt_mean(self, restored_img, target_img):
#         # Tính mean theo từng ảnh trong batch (N, C, H, W) -> (N,)
#         img1_gray = restored_img.mean(dim=1, keepdim=True)
#         img2_gray = target_img.mean(dim=1, keepdim=True)

#         # Scale lại vùng sáng cho khớp với Ground truth
#         mean_restored = img1_gray.mean(dim=[2,3], keepdim=True)
#         mean_target = img2_gray.mean(dim=[2,3], keepdim=True)

#         corrected_img = torch.clamp(restored_img * (mean_target / (mean_restored + 1e-8)), 0, 1)
#         return corrected_img

#     def validation_step(self, batch, batch_idx):
#         low_imgs, high_imgs, img_names = batch
#         enhanced_imgs = self.forward(low_imgs)

#         # Áp dụng chuẩn hóa GT Mean (Bắt buộc với paper LOL Dataset)
#         enhanced_imgs_gt_mean = self.apply_gt_mean(enhanced_imgs, high_imgs)

#         val_psnr = self.psnr(enhanced_imgs_gt_mean, high_imgs)
#         val_ssim = self.ssim(enhanced_imgs_gt_mean, high_imgs)

#         # LPIPS yêu cầu input float tensor
#         # Do normalize=True ở __init__, input [0, 1] là hợp lệ.
#         # Lưu ý: Tính LPIPS có thể hơi tốn resource
#         val_lpips = self.lpips(enhanced_imgs_gt_mean, high_imgs)

#         self.log("val_psnr", val_psnr, on_epoch=True, prog_bar=True, sync_dist=True)
#         self.log("val_ssim", val_ssim, on_epoch=True, prog_bar=True, sync_dist=True)
#         self.log("val_lpips", val_lpips, on_epoch=True, prog_bar=True, sync_dist=True)

#         if batch_idx == 0:
#             self._save_sample_images(low_imgs, enhanced_imgs_gt_mean, high_imgs, stage="val")

#     def test_step(self, batch, batch_idx):
#         low_imgs, high_imgs, img_names = batch
#         enhanced_imgs = self.forward(low_imgs)

#         enhanced_imgs_gt_mean = self.apply_gt_mean(enhanced_imgs, high_imgs)

#         test_psnr = self.psnr(enhanced_imgs_gt_mean, high_imgs)
#         test_ssim = self.ssim(enhanced_imgs_gt_mean, high_imgs)

#         # TÍNH VÀ LOG TEST LPIPS
#         test_lpips = self.lpips(enhanced_imgs_gt_mean, high_imgs)

#         self.log("test_psnr", test_psnr, on_epoch=True, prog_bar=True)
#         self.log("test_ssim", test_ssim, on_epoch=True, prog_bar=True)
#         self.log("test_lpips", test_lpips, on_epoch=True, prog_bar=True)

#         self._save_sample_images(low_imgs, enhanced_imgs_gt_mean, high_imgs, stage="test", img_name=img_names[0])

#     def _save_sample_images(self, low, enh, high, stage, img_name="sample"):
#         out_dir = f"visualize/{stage}"
#         os.makedirs(out_dir, exist_ok=True)
#         grid = torch.cat([low[0], enh[0], high[0]], dim=2)
#         grid_to_log = grid.detach().cpu().clamp(0, 1)
#         filename = f"epoch_{self.current_epoch}" if stage == "val" else img_name

#         vutils.save_image(grid_to_log, os.path.join(out_dir, f"{filename}.png"))

#         if isinstance(self.logger, pl.loggers.WandbLogger):
#             self.logger.experiment.log({
#                 f"Images/{stage}": [wandb.Image(grid_to_log, caption=f"{stage.upper()} - {filename} (Input | Enhanced | GT)")]
#             })

In [ ]:
import os
import torch
import torch.optim as optim
import pytorch_lightning as pl
import torchvision.utils as vutils
# Nếu dùng wandb
try:
    import wandb
except ImportError:
    pass

from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
# THÊM IMPORT CHO LPIPS
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

class LYTLightning(pl.LightningModule):
    def __init__(self, learning_rate=2e-4, max_epochs=1000):
        super().__init__()
        self.save_hyperparameters()

        model = LightweightDualBranchUNet(base_dim=18)
        self.model = torch.compile(model)
        # Loss Model cần device, nên ta khởi tạo lazy hoặc để hook vào Lightning
        self.criterion = None

        self.psnr = PeakSignalNoiseRatio(data_range=1.0)
        self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0)

        # THÊM LPIPS
        # Thông thường net_type = 'vgg' hoặc 'alex' được dùng phổ biến
        # normalize=True giả định input vào khoảng [0, 1] (giống psnr/ssim data_range)
        self.lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True)

    def on_fit_start(self):
        # Khởi tạo loss sau khi module đã được map lên đúng thiết bị GPU
        self.criterion = CombinedLoss(self.device)

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.model.parameters(), lr=self.hparams.learning_rate)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}
        }

    def training_step(self, batch, batch_idx):
        low_imgs, high_imgs, img_names = batch
        enhanced_imgs = self.forward(low_imgs)

        # Loss có giám sát: Giữa ảnh được dự đoán và ảnh GroundTruth (high_imgs)
        loss = self.criterion(high_imgs, enhanced_imgs)

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        low_imgs, high_imgs, img_names = batch
        enhanced_imgs = self.forward(low_imgs)

        # Tính metrics trực tiếp trên ảnh model sinh ra
        val_psnr = self.psnr(enhanced_imgs, high_imgs)
        val_ssim = self.ssim(enhanced_imgs, high_imgs)

        # LPIPS yêu cầu input float tensor
        # Do normalize=True ở __init__, input [0, 1] là hợp lệ.
        val_lpips = self.lpips(enhanced_imgs, high_imgs)

        self.log("val_psnr", val_psnr, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_ssim", val_ssim, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_lpips", val_lpips, on_epoch=True, prog_bar=True, sync_dist=True)

        if batch_idx == 0:
            self._save_sample_images(low_imgs, enhanced_imgs, high_imgs, stage="val")

    def test_step(self, batch, batch_idx):
        low_imgs, high_imgs, img_names = batch
        enhanced_imgs = self.forward(low_imgs)

        # Tính metrics trực tiếp trên ảnh model sinh ra
        test_psnr = self.psnr(enhanced_imgs, high_imgs)
        test_ssim = self.ssim(enhanced_imgs, high_imgs)
        test_lpips = self.lpips(enhanced_imgs, high_imgs)

        self.log("test_psnr", test_psnr, on_epoch=True, prog_bar=True)
        self.log("test_ssim", test_ssim, on_epoch=True, prog_bar=True)
        self.log("test_lpips", test_lpips, on_epoch=True, prog_bar=True)

        self._save_sample_images(low_imgs, enhanced_imgs, high_imgs, stage="test", img_name=img_names[0])

    def _save_sample_images(self, low, enh, high, stage, img_name="sample"):
        out_dir = f"visualize/{stage}"
        os.makedirs(out_dir, exist_ok=True)
        grid = torch.cat([low[0], enh[0], high[0]], dim=2)
        grid_to_log = grid.detach().cpu().clamp(0, 1)
        filename = f"epoch_{self.current_epoch}" if stage == "val" else img_name

        vutils.save_image(grid_to_log, os.path.join(out_dir, f"{filename}.png"))

        if isinstance(self.logger, pl.loggers.WandbLogger):
            self.logger.experiment.log({
                f"Images/{stage}": [wandb.Image(grid_to_log, caption=f"{stage.upper()} - {filename} (Input | Enhanced | GT)")]
            })

# Training

In [ ]:
def main():
    # Khởi tạo seed để dễ dàng tái lập kết quả (reproducibility)
    seed_everything(cfg.seed, workers=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    datamodule = LOLDataModule(
        data_dir=cfg.data_dir,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        patch_size=cfg.patch_size
    )

    model = LYTLightning(learning_rate=cfg.learning_rate, max_epochs=cfg.max_epochs)

    os.makedirs(cfg.ckpt_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=cfg.ckpt_dir,
        filename='lytnet-{epoch:02d}-{val_psnr:.2f}',
        monitor='val_psnr',
        mode='max',
        save_top_k=3,
        save_last=True,
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')

    wandb_logger = WandbLogger(
        project=cfg.project_name,
        name=cfg.run_name,
        save_dir=cfg.work_dir,
        log_model=True
    )

    trainer = pl.Trainer(
        max_epochs=cfg.max_epochs,
        accelerator='auto',
        devices=1,
        logger=wandb_logger,
        callbacks=[checkpoint_callback, lr_monitor],
        log_every_n_steps=10,
        precision=32,
        default_root_dir=cfg.work_dir,
    )

    print(f"🚀 Bắt đầu huấn luyện LYT-Net...")
    trainer.fit(model, datamodule=datamodule)

    print("✨ Đang đánh giá trên tập Test bằng model tốt nhất...")
    trainer.test(datamodule=datamodule, ckpt_path='best')

    wandb.finish()

if __name__ == "__main__":
    main()

Seed set to 1


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


  0%|          | 0.00/528M [00:00<?, ?B/s]

  3%|▎         | 18.4M/528M [00:00<00:02, 192MB/s]

  7%|▋         | 38.6M/528M [00:00<00:02, 204MB/s]

...

W0503 02:43:34.510000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 1. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 76 module(s) in eval mode at
the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore
this warning.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

In [ ]:
# import os
# import torch
# import torchvision.transforms.functional as TF
# from PIL import Image
# import time
# import matplotlib.pyplot as plt  # Import thêm thư viện vẽ hình

# # Giả định bạn đã import kiến trúc mô hình của mình ở đây
# # from your_model_file import LightweightDualBranchUNet

# def infer_single_image_and_show(model_path, image_path, output_path, device="cuda"):
#     """
#     Hàm chạy suy luận cho MỘT ảnh duy nhất, lưu ảnh và hiển thị trực tiếp ra màn hình.
#     """
#     print("⏳ Đang khởi tạo mô hình...")
#     # LƯU Ý: Mở comment dòng dưới và đảm bảo đã import class mô hình của bạn
#     model = LightweightDualBranchUNet(base_dim=18, num_heads=2).to(device)

#     if not os.path.exists(model_path):
#         raise FileNotFoundError(f"Không tìm thấy file trọng số tại: {model_path}")

#     if not os.path.exists(image_path):
#         raise FileNotFoundError(f"Không tìm thấy ảnh đầu vào tại: {image_path}")

#     print(f"📥 Đang nạp trọng số từ: {model_path}")
#     checkpoint = torch.load(model_path, map_location=device)

#     # --- LOGIC BÓC TÁCH VÀ LÀM SẠCH STATE_DICT ---
#     if 'state_dict' in checkpoint:
#         state_dict = checkpoint['state_dict']
#     else:
#         state_dict = checkpoint

#     clean_state_dict = {}
#     for k, v in state_dict.items():
#         if k.startswith('lpips') or k.startswith('criterion'):
#             continue

#         clean_key = k
#         if clean_key.startswith('model.'):
#             clean_key = clean_key.replace('model.', '', 1)
#         elif clean_key.startswith('net.'):
#             clean_key = clean_key.replace('net.', '', 1)
#         if clean_key.startswith('_orig_mod.'):
#             clean_key = clean_key.replace('_orig_mod.', '', 1)

#         clean_state_dict[clean_key] = v

#     # Tắt kiểm tra strict
#     model.load_state_dict(clean_state_dict, strict=False)
#     model.eval()
#     # --------------------------------------------------------------

#     print(f"🖼️ Đang xử lý ảnh: {os.path.basename(image_path)}...")

#     with torch.no_grad():
#         # 1. Đọc ảnh gốc
#         img = Image.open(image_path).convert('RGB')

#         # 2. Resize ảnh về 512x512
#         img_resized = TF.resize(img, [512, 512])

#         # 3. Chuyển thành Tensor
#         input_tensor = TF.to_tensor(img_resized).unsqueeze(0).to(device)

#         # 4. Infer qua model
#         start_time = time.time()
#         # output_tensor = model(input_tensor)

#         # (Dòng test tạm, nhớ xóa khi mở comment chạy model thật)
#         output_tensor = input_tensor

#         end_time = time.time()

#         infer_time = (end_time - start_time) * 1000

#         output_tensor = output_tensor.squeeze(0).cpu()
#         output_tensor = torch.clamp(output_tensor, 0.0, 1.0)

#         # 6. Chuyển tensor về dạng PIL Image
#         out_img = TF.to_pil_image(output_tensor)

#         # --- THÊM CODE LƯU ẢNH Ở ĐÂY ---
#         # Đảm bảo thư mục lưu ảnh tồn tại
#         os.makedirs(os.path.dirname(output_path), exist_ok=True)
#         out_img.save(output_path)
#         print(f"💾 Đã lưu ảnh kết quả tại: {output_path}")
#         # ------------------------------

#         print(f"✅ Xong! Thời gian xử lý: {infer_time:.2f} ms")

#         # --- CODE HIỂN THỊ ẢNH TRỰC TIẾP ---
#         # So sánh ảnh gốc (sau resize) và ảnh đầu ra cạnh nhau
#         fig, axes = plt.subplots(1, 2, figsize=(10, 5))

#         axes[0].imshow(img_resized)
#         axes[0].set_title("Ảnh đầu vào (512x512)")
#         axes[0].axis("off")

#         axes[1].imshow(out_img)
#         axes[1].set_title("Ảnh đầu ra")
#         axes[1].axis("off")

#         plt.tight_layout()
#         plt.show()

# if __name__ == '__main__':
#     WEIGHT_PATH = "/kaggle/input/datasets/vgbao312/finalmodel/lytnet-epoch398-val_psnr23.25.ckpt"
#     IMAGE_PATH = "/kaggle/input/datasets/vgbao312/testimage/test image.jpg"

#     # --- THÊM ĐƯỜNG DẪN LƯU ẢNH ---
#     OUTPUT_PATH = "./output/test_image_enhanced.png"

#     DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"⚙️ Thiết bị suy luận: {DEVICE}")

#     infer_single_image_and_show(WEIGHT_PATH, IMAGE_PATH, OUTPUT_PATH, device=DEVICE)